In [ ]:
import pandas as pd

# Load data
df = pd.read_csv("../data/prices_clean.csv", index_col="Date", parse_dates=True)
metrics = pd.read_csv("../data/metrics_summary.csv", index_col=0)

# Monthly average prices
df.index = pd.to_datetime(df.index)
df["month"] = df.index.to_period("M").astype(str)

monthly_avg = df.groupby("month")[["Gold", "SP500", "BCA", "Telkom", "Bitcoin"]].mean().round(2)
monthly_avg = monthly_avg.reset_index()

print("Monthly Average Prices (first 10 rows):")
print(monthly_avg.head(10))

monthly_avg.to_csv("../data/query1_monthly_avg.csv", index=False)
print("Saved.")

Monthly Average Prices (first 10 rows):
     month     Gold   SP500     BCA   Telkom   Bitcoin
0  2021-01  6180.17  175.02  352.06  2632.74  34967.17
1  2021-02  6032.32  169.34  360.72  2521.30  45773.02
2  2021-03  5860.41  161.25  363.80  2617.41  54630.98
3  2021-04  5571.04  164.94  385.65  2548.82  57357.22
4  2021-05  5723.84  173.34  388.51  2481.85  46754.81
5  2021-06  5684.40  171.56  395.59  2684.07  35960.90
6  2021-07  5405.28  169.06  407.70  2547.55  34368.21
7  2021-08  5724.24  167.11  416.53  2706.00  45388.53
8  2021-09  5872.68  166.04  416.25  2802.50  45292.05
9  2021-10  6583.90  166.13  418.06  3049.75  58173.42
Saved.


In [2]:
# Most volatile months for Bitcoin
btc_monthly = df.groupby("month")["Bitcoin"].agg(
    monthly_high="max",
    monthly_low="min"
).round(2)

btc_monthly["price_range"] = (btc_monthly["monthly_high"] - btc_monthly["monthly_low"]).round(2)
btc_volatile = btc_monthly.sort_values("price_range", ascending=False).head(10).reset_index()

print("Top 10 Most Volatile Months for Bitcoin:")
print(btc_volatile)

btc_volatile.to_csv("../data/query2_btc_volatile.csv", index=False)
print("Saved.")

Top 10 Most Volatile Months for Bitcoin:
     month  monthly_high  monthly_low  price_range
0  2024-11      98997.66     67811.51     31186.15
1  2021-02      55888.13     33537.18     22350.95
2  2021-05      57424.01     35697.61     21726.40
3  2025-11     106547.52     85090.69     21456.83
4  2024-02      62504.79     42658.67     19846.12
5  2025-04      94978.75     76271.95     18706.80
6  2025-10     124752.53    106467.79     18284.74
7  2021-10      65992.84     48116.94     17875.90
8  2025-02     101405.42     84347.02     17058.40
9  2025-05     111673.28     94748.05     16925.23
Saved.


In [3]:
# Asset ranking by Sharpe Ratio
ranking = metrics[["Annualized Return", "Annualized Volatility", "Sharpe Ratio", "Max Drawdown"]].copy()
ranking = ranking.sort_values("Sharpe Ratio", ascending=False)
ranking["Rank"] = range(1, len(ranking) + 1)

print("Asset Ranking by Sharpe Ratio:")
print(ranking)

ranking.to_csv("../data/query3_asset_ranking.csv", index=True)
print("Saved.")

Asset Ranking by Sharpe Ratio:
         Annualized Return  Annualized Volatility  Sharpe Ratio  Max Drawdown  \
SP500               0.1697                 0.1562        0.8304       -0.2103   
BCA                 0.1536                 0.1711        0.6638       -0.2450   
Bitcoin             0.3773                 0.5874        0.5742       -0.7663   
Gold                0.0843                 0.2263        0.1958       -0.3141   
Telkom              0.0918                 0.2829        0.1830       -0.4669   

         Rank  
SP500       1  
BCA         2  
Bitcoin     3  
Gold        4  
Telkom      5  
Saved.


In [4]:
import os

os.makedirs("../sql", exist_ok=True)

sql1 = """
SELECT
    strftime('%Y-%m', Date) AS month,
    ROUND(AVG(Bitcoin), 2) AS avg_bitcoin,
    ROUND(AVG(SP500), 2) AS avg_sp500,
    ROUND(AVG(Gold), 2) AS avg_gold,
    ROUND(AVG(BCA), 2) AS avg_bca,
    ROUND(AVG(Telkom), 2) AS avg_telkom
FROM prices
GROUP BY month
ORDER BY month;
"""

sql2 = """
SELECT
    strftime('%Y-%m', Date) AS month,
    ROUND(MAX(Bitcoin), 2) AS monthly_high,
    ROUND(MIN(Bitcoin), 2) AS monthly_low,
    ROUND(MAX(Bitcoin) - MIN(Bitcoin), 2) AS price_range
FROM prices
GROUP BY month
ORDER BY price_range DESC
LIMIT 10;
"""

sql3 = """
SELECT
    asset,
    ROUND(annualized_return, 4) AS annual_return,
    ROUND(annualized_volatility, 4) AS volatility,
    ROUND(sharpe_ratio, 4) AS sharpe_ratio,
    ROUND(max_drawdown, 4) AS max_drawdown,
    RANK() OVER (ORDER BY sharpe_ratio DESC) AS rank
FROM metrics
ORDER BY sharpe_ratio DESC;
"""

with open("../sql/query1_monthly_avg.sql", "w") as f:
    f.write(sql1)

with open("../sql/query2_btc_volatile.sql", "w") as f:
    f.write(sql2)

with open("../sql/query3_asset_ranking.sql", "w") as f:
    f.write(sql3)

print("SQL files saved to sql/ folder")

SQL files saved to sql/ folder
